##### Sales
* assumes that delivery orders are taken by e-commerce
* time series
* all sales
* daily change in sales

In [1]:
import polars as pl

In [2]:
import os

datafiles = os.listdir("../data")
datafiles

['sellers.csv',
 'products.csv',
 'customers.csv',
 'order_items.csv',
 'category_name_translation.csv',
 'order_reviews.csv',
 'geolocation.csv',
 'payments.csv',
 'schema.png',
 'orders.csv']

In [3]:
orders = pl.read_csv(source = "../data/orders.csv",
                    schema_overrides = {"order_purchase_timestamp": pl.Datetime,
                                        "order_approved_at": pl.Datetime,
                                        "order_delivered_carrier_date": pl.Datetime,
                                        "order_delivered_customer_date": pl.Datetime,
                                        "order_estimated_delivery_date": pl.Datetime})
orders

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""",2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""",2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
"""949d5b44dbf5de918fe9c16f97b45f…","""f88197465ea7920adcdbec7375364d…","""delivered""",2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
"""ad21c59c0840e6cb83a9ceb5573f81…","""8ab97904e6daea8866dbdbc4fb7aad…","""delivered""",2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
…,…,…,…,…,…,…,…
"""9c5dedf39a927c1b2549525ed64a05…","""39bd1228ee8140590ac3aca26f2dfe…","""delivered""",2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
"""63943bddc261676b46f01ca7ac2f7b…","""1fca14ff2861355f6e5f14306ff977…","""delivered""",2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
"""83c1379a015df1e13d02aae0204711…","""1aa71eb042121263aafbe80c1b562c…","""delivered""",2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00


In [4]:
items = pl.read_csv("../data/order_items.csv",
                    schema_overrides = {"shipping_limit_date": pl.Datetime})

In [5]:
items

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""00010242fe8c5a6d1ba2dd792cb162…",1,"""4244733e06e7ecb4970a6e2683c13e…","""48436dade18ac8b2bce089ec2a0412…",2017-09-19 09:45:35,58.9,13.29
"""00018f77f2f0320c557190d7a144bd…",1,"""e5f2d52b802189ee658865ca93d83a…","""dd7ddc04e1b6c2c614352b383efe2d…",2017-05-03 11:05:13,239.9,19.93
"""000229ec398224ef6ca0657da4fc70…",1,"""c777355d18b72b67abbeef9df44fd0…","""5b51032eddd242adc84c38acab88f2…",2018-01-18 14:48:30,199.0,17.87
"""00024acbcdf0a6daa1e931b038114c…",1,"""7634da152a4610f1595efa32f14722…","""9d7a1d34a5052409006425275ba1c2…",2018-08-15 10:10:18,12.99,12.79
"""00042b26cf59d7ce69dfabb4e55b4f…",1,"""ac6c3623068f30de03045865e4e100…","""df560393f3a51e74553ab94004ba5c…",2017-02-13 13:57:51,199.9,18.14
…,…,…,…,…,…,…
"""fffc94f6ce00a00581880bf54a75a0…",1,"""4aa6014eceb682077f9dc4bffebc05…","""b8bc237ba3788b23da09c0f1f3a328…",2018-05-02 04:11:01,299.99,43.41
"""fffcd46ef2263f404302a634eb57f7…",1,"""32e07fd915822b0765e448c4dd74c8…","""f3c38ab652836d21de61fb8314b691…",2018-07-20 04:31:48,350.0,36.53
"""fffce4705a9662cd70adb13d4a3183…",1,"""72a30483855e2eafc67aee5dc25604…","""c3cfdc648177fdbbbb35635a37472c…",2017-10-30 17:14:25,99.9,16.95


In [9]:
sales = items.join(orders,
          on = "order_id",
          how = "left")
sales = sales.with_columns(
    pl.col("order_purchase_timestamp").dt.month().alias("month"),
    pl.col("order_purchase_timestamp").dt.year().alias("year")
)
monthly_sales = sales.group_by(["year", "month"]).agg(pl.col("freight_value").sum().alias("sales")).sort(["year", "month"])

In [7]:
monthly_sales = monthly_sales.with_columns(pl.col("sales").diff().alias("growth"))
monthly_sales = monthly_sales.with_columns(pl.col("sales").diff(n=12).alias("y_y_change"))

In [17]:
monthly_sales.with_columns(pl.date(2018, pl.col("month"), 1).dt.strftime("%B").alias("month"))

year,month,sales
i32,str,f64
2016,"""September""",87.39
2016,"""October""",7301.18
2016,"""December""",8.72
2017,"""January""",16875.62
2017,"""February""",38977.6
…,…,…
2018,"""May""",153264.14
2018,"""June""",157552.8
2018,"""July""",163220.81
